# Follow-ups: R5 control, persistence by transfer, who turns over, and the spatial program

Continues `relationships_iso.ipynb`. Two mouse splits are used: the original one for the state experiments (E1–E3
use all mice where a result is a control or a transfer), and a **new random split made before any spatial
analysis** for the spatial questions (`data/processed/mouse_split.json`, key `spatial`: discovery 688277, 692644,
703063, 709401; confirmation 703064, 703065, 720762).

**Data expansion (Step 0).** Envelopes for every ~40 µm along all 29 probes (2,361 channels used here) and all
valid EEG electrodes, awake and stimulation-free isoflurane seconds (`src/multichannel.py`), in two versions:
*as recorded*, and *local* — current source density (second spatial difference) along each probe and a common
average reference for the EEG. All 3,730 well-isolated grey-matter units are used (≥ 0.5 Hz awake).

Code: `src/r5_control.py`, `src/transfer_states.py`, `src/turnover.py`, `src/locality.py`, `src/cross_region.py`,
`src/position_pole.py`, `src/spatial_maps.py` (runners `src/run_*.py`).

In [1]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import wilcoxon
warnings.filterwarnings("ignore")
REPO = Path("/Users/namanrudrakshi/Developer/dc/dc-allen-data-acquire")
sys.path.insert(0, str(REPO / "code" / "python"))
from src import position_pole as pp
PI, PS = REPO / "data/processed/relationships_iso", REPO / "data/processed/spatial"
OUT = REPO / "data" / "out" / "2026-11-16_spatial_followups"; FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "0.4", "axes.labelcolor": "0.15", "xtick.color": "0.3", "ytick.color": "0.3"})
def save(fig, name): fig.savefig(FIG / f"{name}.png", dpi=200, bbox_inches="tight", facecolor="white")
SP = json.load(open(REPO / "data/processed/mouse_split.json"))["spatial"]
grp = lambda s: np.where(pd.Series(s).str[:6].isin(SP["discovery"]), "discovery", "confirmation")
AW, ISO = "#2a78d6", "#e34948"

## E1. Is R5 (pole separation ↔ coupling strength) about the motif?
R5 was the one relationship confirmed in both states. Here every 2-min window's pole separation and neuron
coupling are recomputed along other directions, and with spikes shifted so they have no true relation to the LFP
(what remains is the pure "more LFP variance → higher correlations" effect). Partial Spearman r across windows.

In [2]:
D = pd.read_pickle(PI / "r5_control.pkl")
D["kind"] = D.direction.where(~D.direction.str.startswith("random"), "random (50)")
E1 = D.groupby(["state", "spikes", "kind"])[["sep~corr", "sep~slope"]].median().round(3)
E1.to_csv(OUT / "E1_r5_control.csv"); E1

sep~corr  sep~slope
state      spikes                     kind                            
awake      real                       band PC1        0.473     -0.132
                                      broadband       0.400     -0.295
                                      motif           0.534     -0.134
                                      random (50)     0.322     -0.156
           shifted (no true coupling) band PC1        0.134     -0.504
                                      broadband       0.168     -0.589
                                      motif           0.240     -0.497
isoflurane real                       band PC1        0.567      0.075
                                      broadband       0.561     -0.052
                                      motif           0.500      0.152
                                      random (50)     0.373      0.190
           shifted (no true coupling) band PC1        0.114     -0.318
                                      broadband       0.171     -0.343
                                      motif           0.126     -0.291

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
kinds = ["motif", "band PC1", "broadband", "random (50)"]
for ax, st in zip(axes, ("awake", "isoflurane")):
    real = D[(D.state == st) & (D.spikes == "real")].groupby("kind")["sep~corr"].median().reindex(kinds)
    sh = D[(D.state == st) & (D.spikes != "real")].groupby("kind")["sep~corr"].median().reindex(kinds)
    ax.bar(np.arange(4) - 0.18, real, 0.34, color=AW if st == "awake" else ISO, label="real spikes")
    ax.bar(np.arange(4) + 0.18, sh, 0.34, color="0.75", label="spikes shifted (artefact only)")
    ax.set_xticks(range(4), kinds); ax.set_title(st, fontsize=9, loc="left"); ax.axhline(0, color="0.6", lw=0.6)
axes[0].set_ylabel("separation ↔ coupling\n(median partial r over signals)"); axes[0].legend(frameon=False, fontsize=8)
fig.suptitle("E1: the relation is real (well above the artefact) but not specific to the motif", fontsize=10, x=0.01, ha="left")
fig.tight_layout(); save(fig, "E1_r5_control"); plt.show()

## E2. Do the tracker → spiking mappings persist when fitted in one state and applied in the other?

In [4]:
T = pd.read_pickle(PI / "transfer_states.pkl")
E2 = T.groupby(["direction", "frame", "target"]).agg(signals=("session", "size"), transfer_r=("transfer_r", "median"),
                                                     within_state_r=("ceiling_r", "median"),
                                                     beats_null=("p", lambda p: f"{(p < .05).sum()}/{p.notna().sum()}" if p.notna().any() else "")).round(3)
E2.to_csv(OUT / "E2_transfer.csv"); E2

signals  \
direction           frame          target                             
awake -> isoflurane awake units    + minus - contrast            33   
                                   population rate               35   
                                   single neurons (median)       35   
                    within-state z + minus - contrast            33   
                                   population rate               35   
                                   single neurons (median)       35   
isoflurane -> awake awake units    + minus - contrast            33   
                                   population rate               35   
                                   single neurons (median)       35   
                    within-state z + minus - contrast            33   
                                   population rate               35   
                                   single neurons (median)       35   

                                                            transfer_r  \
direction           frame          target                                
awake -> isoflurane awake units    + minus - contrast           -0.006   
                                   population rate               0.006   
                                   single neurons (median)      -0.001   
                    within-state z + minus - contrast            0.058   
                                   population rate               0.029   
                                   single neurons (median)       0.007   
isoflurane -> awake awake units    + minus - contrast            0.110   
                                   population rate               0.023   
                                   single neurons (median)       0.001   
                    within-state z + minus - contrast            0.124   
                                   population rate               0.046   
                                   single neurons (median)       0.004   

                                                            within_state_r  \
direction           frame          target                                    
awake -> isoflurane awake units    + minus - contrast                0.158   
                                   population rate                   0.623   
                                   single neurons (median)           0.341   
                    within-state z + minus - contrast                0.158   
                                   population rate                   0.623   
                                   single neurons (median)           0.341   
isoflurane -> awake awake units    + minus - contrast                0.708   
                                   population rate                   0.431   
                                   single neurons (median)           0.335   
                    within-state z + minus - contrast                0.708   
                                   population rate                   0.431   
                                   single neurons (median)           0.335   

                                                           beats_null  
direction           frame          target                              
awake -> isoflurane awake units    + minus - contrast           13/33  
                                   population rate              13/35  
                                   single neurons (median)             
                    within-state z + minus - contrast           12/33  
                                   population rate              12/35  
                                   single neurons (median)             
isoflurane -> awake awake units    + minus - contrast           17/33  
                                   population rate              14/35  
                                   single neurons (median)             
                    within-state z + minus - contrast           16/33  
                                   population rate              11/35  

## E3. Who joins or leaves between awake and isoflurane? (spatial split; depth is a spatial predictor)

In [5]:
E3 = pd.read_pickle(PI / "turnover_predictors.pkl")
E3[["group", "comparison", "predictor", "difference (SD)", "p"]].round(3).to_csv(OUT / "E3_turnover_predictors.csv", index=False)
E3.pivot_table(index=["comparison", "predictor"], columns="group", values=["difference (SD)", "p"]).round(3)

difference (SD)                      p  \
group                             confirmation discovery confirmation   
comparison      predictor                                               
joined vs never abs_r_awake              0.059    -0.127        0.622   
                cv2                     -0.082     0.009        0.467   
                log_rate_awake          -0.052     0.114        0.658   
                log_rate_ratio           0.101     0.158        0.363   
                narrow                  -0.004     0.005        0.970   
                pop_coupling             0.055     0.025        0.636   
                rel_depth               -0.172     0.033        0.140   
left vs stayed  abs_r_awake             -0.326    -0.158        0.011   
                cv2                      0.150     0.111        0.218   
                log_rate_awake          -0.210     0.073        0.092   
                log_rate_ratio           0.066    -0.043        0.594   
                narrow                  -0.015     0.046        0.881   
                pop_coupling            -0.093     0.036        0.451   
                rel_depth                0.056     0.128        0.656   

                                          
group                          discovery  
comparison      predictor                 
joined vs never abs_r_awake        0.207  
                cv2                0.931  
                log_rate_awake     0.247  
                log_rate_ratio     0.110  
                narrow             0.961  
                pop_coupling       0.807  
                rel_depth          0.759  
left vs stayed  abs_r_awake        0.180  
                cv2                0.350  
                log_rate_awake     0.521  
                log_rate_ratio     0.735  
                narrow             0.663  
                pop_coupling       0.764  
                rel_depth          0.273

## S-A. How local is the coupling between tracker signals and single neurons?
|r| between each unit's rate (population mean removed) and each channel's signal, minus the shift null, by distance.
As recorded, the LFP is nearly the same along the whole probe (signal spread and the shared reference), so it
cannot show locality. With local referencing (CSD) the LFP decorrelates within ~100 µm.

In [6]:
curves = {}
for folder, lab in (("multichannel", "as recorded"), ("multichannel_local", "local (CSD / CAR)")):
    Tl = pd.read_pickle(PS / f"locality_{folder}.pkl"); Tl = Tl[(Tl.pair == "same probe") & (Tl.rates == "population removed")]
    Tl["excess"] = Tl.abs_r - Tl.null_abs_r
    curves[lab] = Tl.groupby(["state", "signal", "bin_lo"]).apply(lambda d: np.average(d.excess, weights=d.n_pairs)).unstack("bin_lo")
    curves[lab + " LFP"] = pd.read_pickle(PS / f"lfp_decay_{folder}.pkl").groupby(["state", "bin_lo"]).lfp_abs_r.mean().unstack()
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
bins = [0, 50, 100, 200, 400, 800, 1600]; xl = ["0–50", "50–100", "100–200", "200–400", "400–800", "0.8–1.6 mm", ">1.6 mm"]
for ax, lab in zip(axes[:2], ("as recorded", "local (CSD / CAR)")):
    for st, col in (("awake", AW), ("isoflurane", ISO)):
        for sig, ls in (("fast", "-"), ("pc1", "--"), ("slow", ":")):
            y = curves[lab].loc[(st, sig)].reindex(bins)
            ax.plot(range(7), y, ls, color=col, lw=1.6, label=f"{st}, {sig}" if lab == "as recorded" else None)
    ax.set_xticks(range(7), xl, rotation=30, fontsize=8); ax.set_title(f"{lab}: coupling vs depth distance", fontsize=9, loc="left")
    ax.set_ylabel("|r| above null")
axes[0].legend(frameon=False, fontsize=7.5, ncol=2)
ax = axes[2]
for lab, ls in (("as recorded LFP", "-"), ("local (CSD / CAR) LFP", "--")):
    for st, col in (("awake", AW), ("isoflurane", ISO)):
        ax.plot(range(7), curves[lab].loc[st].reindex(bins), ls, color=col, label=f"{lab.replace(' LFP', '')}, {st}")
ax.set_xticks(range(7), xl, rotation=30, fontsize=8); ax.set_title("the LFP's own decorrelation with distance", fontsize=9, loc="left")
ax.set_ylabel("|r| between channels"); ax.legend(frameon=False, fontsize=7.5)
fig.tight_layout(); save(fig, "SA_locality"); plt.show()
L = pd.read_pickle(PS / "locality_index.pkl")
SA = L[L.signal.isin(["fast", "fast<30", "pc1", "slow"])].groupby(["reference", "group", "state", "signal"]).apply(lambda d: pd.Series({
    "sessions": d.locality.notna().sum(), "near (≤100 µm)": d.near.median(), "far (≥800 µm)": d.far.median(),
    "locality (near−far)/near": d.locality.median(), "near > far": f"{(d.near > d.far).sum()}/{d[['near', 'far']].dropna().shape[0]}"}))
SA.to_csv(OUT / "SA_locality_index.csv"); SA.round(3)

sessions  near (≤100 µm)  \
reference   group        state      signal                              
as recorded confirmation awake      fast            3           0.110   
                                    fast<30         3           0.101   
                                    pc1             3           0.105   
                                    slow            3           0.056   
                         isoflurane fast            3           0.062   
                                    fast<30         3           0.059   
                                    pc1             3           0.050   
                                    slow            2           0.006   
            discovery    awake      fast            4           0.076   
                                    fast<30         5           0.071   
                                    pc1             4           0.068   
                                    slow            5           0.104   
                         isoflurane fast            4           0.079   
                                    fast<30         4           0.074   
                                    pc1             4           0.076   
                                    slow            2          -0.000   
local (CSD) confirmation awake      fast            3           0.042   
                                    fast<30         3           0.046   
                                    pc1             3           0.048   
                                    slow            3           0.037   
                         isoflurane fast            3           0.037   
                                    fast<30         3           0.043   
                                    pc1             3           0.043   
                                    slow            3           0.028   
            discovery    awake      fast            4           0.042   
                                    fast<30         4           0.052   
                                    pc1             4           0.048   
                                    slow            5           0.065   
                         isoflurane fast            4           0.051   
                                    fast<30         4           0.059   
                                    pc1             4           0.051   
                                    slow            4           0.016   

                                             far (≥800 µm)  \
reference   group        state      signal                   
as recorded confirmation awake      fast             0.103   
                                    fast<30          0.091   
                                    pc1              0.102   
                                    slow             0.065   
                         isoflurane fast             0.049   
                                    fast<30          0.047   
                                    pc1              0.038   
                                    slow            -0.005   
            discovery    awake      fast             0.065   
                                    fast<30          0.058   
                                    pc1              0.064   
                                    slow             0.101   
                         isoflurane fast             0.079   
                                    fast<30          0.074   
                                    pc1              0.074   
                                    slow             0.004   
local (CSD) confirmation awake      fast             0.032   
                                    fast<30          0.034   
                                    pc1              0.033   
                                    slow             0.045   
                         isoflurane fast             0.030   
                                    fast<30          0.032   
                                    pc1              0.031   
            

## S-B. Do ensembles stay inside a region or probe?
Purity = share of an ensemble's members in its most common region (probe); null = labels permuted across units.

In [7]:
X = pd.read_pickle(PS / "cross_region.pkl"); X["group"] = grp(X.session); X["excess"] = X.purity - X.null_purity
SB = X.groupby(["group", "method", "by", "state"]).agg(ensembles=("excess", "size"), size=("size", "median"),
                                                      regions_spanned=("n_distinct", "median"), purity=("purity", "median"),
                                                      null=("null_purity", "median"), excess=("excess", "median"),
                                                      more_confined_than_chance=("p", lambda p: f"{(p < .05).sum()}/{len(p)}"))
SB.to_csv(OUT / "SB_cross_region.csv"); SB.round(3)

ensembles  size  regions_spanned  \
group        method   by     state                                          
confirmation corr     probe  awake              15  28.0              4.0   
                             isoflurane         15  81.0              4.0   
                      region awake              15  28.0             15.0   
                             isoflurane         15  81.0             19.0   
             getclust probe  awake              22   6.0              3.0   
                             isoflurane         12  10.0              3.5   
                      region awake              22   6.0              5.0   
                             isoflurane         12  10.0              6.0   
discovery    corr     probe  awake              17  86.0              3.0   
                             isoflurane         26  52.5              3.0   
                      region awake              17  86.0             17.0   
                             isoflurane         26  52.5             12.0   
             getclust probe  awake              31   7.0              3.0   
                             isoflurane         19  10.0              3.0   
                      region awake              31   7.0              5.0   
                             isoflurane         19  10.0              5.0   

                                         purity   null  excess  \
group        method   by     state                               
confirmation corr     probe  awake        0.478  0.443   0.082   
                             isoflurane   0.439  0.369   0.070   
                      region awake        0.208  0.163   0.060   
                             isoflurane   0.200  0.143   0.046   
             getclust probe  awake        0.600  0.492   0.044   
                             isoflurane   0.486  0.465   0.014   
                      region awake        0.354  0.280   0.086   
                             isoflurane   0.259  0.234   0.058   
discovery    corr     probe  awake        0.556  0.559   0.082   
                             isoflurane   0.579  0.521   0.107   
                      region awake        0.229  0.210   0.017   
                             isoflurane   0.343  0.241   0.099   
             getclust probe  awake        0.600  0.562   0.059   
                             isoflurane   0.571  0.561   0.037   
                      region awake        0.400  0.303   0.086   
                             isoflurane   0.400  0.288   0.041   

                                        more_confined_than_chance  
group        method   by     state                                 
confirmation corr     probe  awake                           4/15  
                             isoflurane                      6/15  
                      region awake                           5/15  
                             isoflurane                      7/15  
             getclust probe  awake                           3/22  
                             isoflurane                      1/12  
                      region awake                           1/22  
                             isoflurane                      2/12  
discovery    corr     probe  awake                           6/17  
                             isoflurane                     14/26  
                      region awake                           8/17  
                             isoflurane                     16/26  
             getclust probe  awake                           6/31  
                             isoflurane                      2/19  
                      region awake                           8/31  
                             isoflurane                      4/19

## S-C. Does position across the whole probe decide a neuron's pole or its turnover?
Local signal = the CSD channel nearest each unit. Depth effects are within-probe standardized (+ = deeper); layer
statistic = spread of the share across cortical layers. Labels permuted within probe. Discovery → confirmation.

In [8]:
SC = pd.read_pickle(PS / "position_pole_tests.pkl")
SCw = SC.pivot_table(index="test", columns="group", values=["effect", "p"]).round(4)
SCw.to_csv(OUT / "SC_position_pole.csv"); SCw

effect                      p          
group                     confirmation discovery confirmation discovery
test                                                                   
awake pole + vs - ~ depth       0.0481    0.4356       0.6677    0.0005
awake pole + vs - ~ layer       1.9900    2.4150       0.1554    0.0890
iso pole + vs - ~ depth         0.4189    0.3801       0.0020    0.0030
iso pole + vs - ~ layer         1.9215    1.2072       0.1654    0.5577
joined vs never ~ depth         0.0179    0.1142       0.8206    0.0910
joined vs never ~ layer         3.1545    2.7310       0.0010    0.0415
left vs stayed ~ depth         -0.1555   -0.7425       0.3028    0.0005
left vs stayed ~ layer          2.9064    0.6729       0.0140    0.8781

## S-D. Where do the tracker's state changes happen?
Per local channel: baseline shift (isoflurane − awake, awake units), its projection on the motif and on broadband,
pole-separation and spread ratios. Reproducibility across mice: region (layer) means in discovery vs confirmation
mice, Spearman r; null permutes region labels within mouse.

In [9]:
from src import spatial_maps as sm
C = pd.read_pickle(PS / "channel_changes.pkl"); C["mouse"] = C.session.str[:6]; C["layer"] = C.region.map(pp.layer_of)
P, Ee = C[C.array != "EEG"], C[C.array == "EEG"]
rng = np.random.default_rng(0); rows = []
measures = ["shift_size", "shift_on_motif", "shift_on_broadband", "separation_ratio", "spread_ratio"]
for m in measures:
    rows.append({**sm.reproducibility(P, "region", m, SP, rng), "where": "probe channels, by region"})
    rows.append({**sm.reproducibility(P[P.layer != "subcortical"], "layer", m, SP, rng), "where": "probe channels, by cortical layer"})
    rows.append({**sm.reproducibility(Ee, "region", m, SP, rng), "where": "EEG electrodes, by area"})
SD = pd.DataFrame(rows)[["where", "measure", "shared", "spearman_r", "p"]]; SD.to_csv(OUT / "SD_map_reproducibility.csv", index=False)
fig, ax = plt.subplots(figsize=(7, 3.4))
order = ["1", "2/3", "4", "5", "6a", "6b"]
for g, col in (("discovery", "#2a78d6"), ("confirmation", "#1baf7a")):
    v = P[(P.layer != "subcortical") & P.mouse.isin(SP[g])].groupby("layer").shift_on_broadband.median().reindex(order)
    ax.plot(range(6), v, "-o", color=col, label=f"{g} mice")
ax.set_xticks(range(6), [f"L{l}" for l in order]); ax.axhline(0, color="0.6", lw=0.6)
ax.set_ylabel("baseline shift on broadband\n(isoflurane − awake, awake SDs)"); ax.legend(frameon=False, fontsize=8)
ax.set_title("S-D: the isoflurane baseline drop by cortical layer (local channels)", fontsize=9, loc="left")
save(fig, "SD_layer_profile"); plt.show()
SD.round(3)

,where,measure,shared,spearman_r,p
0,"probe channels, by region",shift_size,49,0.371,0.002
1,"probe channels, by cortical layer",shift_size,6,0.600,0.097
2,"EEG electrodes, by area",shift_size,9,0.350,0.203
3,"probe channels, by region",shift_on_motif,49,0.484,0.001
4,"probe channels, by cortical layer",shift_on_motif,6,0.771,0.067
5,"EEG electrodes, by area",shift_on_motif,9,0.467,0.151
6,"probe channels, by region",shift_on_broadband,49,0.458,0.003
7,"probe channels, by cortical layer",shift_on_broadband,6,0.486,0.172
8,"EEG electrodes, by area",shift_on_broadband,9,0.717,0.021
9,"probe channels, by region",separation_ratio,49,0.334,0.003


## S-E. Do EEG electrode changes track the neural changes beneath them?
Per EEG electrode (common-average referenced): state changes vs the median firing-rate change and the turnover share
of units within 1 mm horizontally. Spearman across electrodes within mouse, averaged; null permutes electrodes.

In [10]:
SE = pd.read_pickle(PS / "eeg_vs_beneath_tests.pkl"); SE.to_csv(OUT / "SE_eeg_vs_beneath.csv", index=False)
SE.pivot_table(index=["EEG change", "neural change beneath"], columns="group", values=["mean r", "p"]).round(3)

mean r                      p  \
group                                    confirmation discovery confirmation   
EEG change         neural change beneath                                       
separation_ratio   rate_change                  0.111    -0.129        0.587   
                   turnover                     0.290    -0.004        0.263   
shift_on_broadband rate_change                  0.312    -0.305        0.150   
                   turnover                     0.195     0.201        0.469   
shift_on_motif     rate_change                  0.032    -0.369        0.878   
                   turnover                    -0.432     0.194        0.108   
shift_size         rate_change                 -0.019    -0.404        0.910   
                   turnover                    -0.405     0.070        0.114   
spread_ratio       rate_change                 -0.012    -0.275        0.954   
                   turnover                     0.563    -0.087        0.038   

                                                    
group                                    discovery  
EEG change         neural change beneath            
separation_ratio   rate_change               0.519  
                   turnover                  0.994  
shift_on_broadband rate_change               0.102  
                   turnover                  0.521  
shift_on_motif     rate_change               0.040  
                   turnover                  0.489  
shift_size         rate_change               0.030  
                   turnover                  0.790  
spread_ratio       rate_change               0.140  
                   turnover                  0.752

## S-D (shape). Does the motif itself change with depth, layer or region?
Per-channel awake motif libraries (every ~160 µm on probes, every EEG electrode). Pair similarity = best sign-free
match between two channels' libraries. Within probe: Spearman r of similarity vs depth distance, null permutes depths
within probe, pooled over probes per group. Layer / region profiles: canonicality (|cos| of the channel's best motif
with the other mice's consensus), library size and band loadings, discovery vs confirmation reproducibility.
Caveat: libraries come from the as-recorded LFP, which is near-global along a probe, so differences are understated.

In [11]:
from src import motif_shape as ms
MS = pd.read_pickle(PS / "motif_shape.pkl")
Dt = MS["distance"]
print({g: ms.pooled(Dt, SP[g]) for g in ("discovery", "confirmation")})
Tsh = ms.channel_table()
fig, ax = plt.subplots(figsize=(6, 3.4))
bins = np.arange(0, 3600, 400)
for g, col in (("discovery", "#2a78d6"), ("confirmation", "#1baf7a")):
    dd, ss = [], []
    for (_, _), P_ in Tsh[(~Tsh.is_eeg) & (Tsh.n_motifs > 0) & Tsh.mouse.isin(SP[g])].groupby(["session", "array"]):
        L, z = P_.U.to_list(), P_.depth_um.to_numpy()
        for i in range(len(L)):
            for j in range(i + 1, len(L)):
                dd.append(abs(z[i] - z[j])); ss.append(ms.pair_sim(L[i], L[j]))
    dd, ss = np.array(dd), np.array(ss); k = np.digitize(dd, bins)
    ax.plot([bins[i - 1] + 200 for i in np.unique(k)], [ss[k == i].mean() for i in np.unique(k)], "-o", color=col, label=f"{g} mice")
ax.set_xlabel("depth distance between channels (µm)"); ax.set_ylabel("motif similarity (best |cos|)")
ax.legend(frameon=False, fontsize=8); ax.set_title("S-D shape: channel motifs diverge with depth distance", fontsize=9, loc="left")
save(fig, "SD_shape_distance"); plt.show()
MS["repro"].to_csv(OUT / "SD_shape_reproducibility.csv", index=False)
Dt.drop(columns="null").to_csv(OUT / "SD_shape_distance.csv", index=False)
display(MS["layer"].round(2)); MS["repro"].round(3)

{'discovery': {'probes': 11, 'mean_r': -0.41429429749839974, 'p_lower': 0.000999000999000999, 'near_minus_far': 0.0564266038099289}, 'confirmation': {'probes': 12, 'mean_r': -0.3180828746672381, 'p_lower': 0.000999000999000999, 'near_minus_far': 0.037652553269971224}}


,canonical,n_motifs,band0,band1,band2,band3,band4,band5,band6,band7,band8,band9,n
layer,,,,,,,,,,,,,
1,0.71,1.33,0.23,0.27,0.10,0.35,0.38,0.37,0.28,0.18,-0.04,-0.10,55
2/3,0.73,1.25,0.22,0.26,0.16,0.36,0.37,0.38,0.32,0.24,0.06,-0.07,56
4,0.74,1.62,0.20,0.25,0.13,0.36,0.38,0.36,0.32,0.23,0.11,0.00,16
5,0.80,1.25,0.19,0.25,0.19,0.36,0.36,0.36,0.30,0.19,0.02,-0.06,67
6a,0.77,1.29,0.24,0.24,0.18,0.34,0.35,0.35,0.32,0.16,-0.06,-0.13,63
6b,0.83,1.00,0.15,0.18,0.17,0.38,0.39,0.42,0.35,0.27,0.10,-0.07,6


,measure,by,shared,spearman_r,p
0,canonical,layer,5,-0.700,0.920
1,n_motifs,layer,5,0.205,0.370
2,band0,layer,5,-0.500,0.804
3,band1,layer,5,-0.600,0.865
4,band2,layer,5,0.900,0.066
5,band3,layer,5,-0.400,0.782
6,band4,layer,5,0.200,0.404
7,band5,layer,5,-0.700,0.924
8,band6,layer,5,-0.200,0.588
9,band7,layer,5,-0.400,0.812


## Conclusions
**State experiments**
- **E1.** R5 is real (well above the pure-variance artefact in both states) but not motif-specific: band PC1 and
  broadband show it as strongly. Theme: when the LFP's main axis swings more, neurons lock to it more tightly.
- **E2.** Fitted mappings do not transfer between states (median transfer r −0.01 to 0.12 vs within-state 0.16–0.71;
  sign differs between mice). What persists is the kind of relationship, not its weights.
- **E3.** Nothing tested (cell type, rate, rate change, burstiness, population coupling, depth) predicts which neurons
  join or leave.

**Spatial program**
- **S-A.** As recorded, the LFP is near-identical along the probe and coupling does not fall with distance. With
  local referencing (CSD) coupling is stronger within 100 µm than beyond 800 µm in every session (7/7 for the fast
  signal, both states; ~25–55 % higher), including with bands ≤ 30 Hz only (not spike leakage). Isoflurane is not
  consistently more local than awake (2/3 sessions per group).
- **S-B.** Ensembles are distributed: 28–86-neuron communities span 12–19 regions, getClust groups ~5; purity only
  slightly above chance; no consistent state change.
- **S-C.** Confirmed: under isoflurane, neurons following the motif's + pole sit deeper than − pole neurons
  (+0.38 SD discovery, +0.42 SD confirmation, p = 0.002). Awake depth effect and "leavers are shallower" did not
  confirm. Exploratory (confirmation only): layer 1 units join more under isoflurane.
- **S-D.** Where the state change happens is reproducible across independent mice: region maps of the baseline
  shift (size, along the motif, along broadband) and of pole separation correlate between discovery and confirmation
  mice (r 0.33–0.48, p ≤ 0.005; 49 regions). Layer 5 shows the largest isoflurane baseline drop. EEG: the broadband
  shift map is reproducible across mice (r 0.72, p = 0.03, 9 areas).
- **S-E.** EEG electrode changes do not reliably track firing changes beneath them (a discovery-only hint, r −0.40,
  p = 0.03, not corrected and not confirmed).

- **S-D shape.** Motif shape changes gradually along a probe: channel motifs become less similar with depth distance
  in both mouse groups (mean within-probe r −0.41 discovery, −0.32 confirmation, both p = 0.001; near ≤ 400 µm vs
  far ≥ 1.6 mm similarity +0.04–0.06), even on the as-recorded LFP. But no layer or region profile of motif shape
  (canonicality, library size, band loadings) reproduces across mice: the drift is gradual, not a layer signature.
  About 30 % of probe channels and 43 % of EEG electrodes have too little awake data for an own library.